In [ ]:
# ==============================================
# Step 1: Import Required Libraries & Connect to WRDS
# ==============================================

# Connect to WRDS database
import wrds
import pandas as pd
username = "yutongzzz"
db = wrds.Connection(wrds_username=username)

# Define target companies and analysis period
COMPANIES = {
    "002594": "BYD",
    "601633": "Great Wall",
    "000625": "Changan"
}
START_YEAR = 2020

# ==============================================
# Step 2: Load Raw Financial Data & Data Cleaning
# ==============================================

# Get stock list for SQL query
stock_list = tuple(COMPANIES.keys())

# SQL query to extract core financial indicators 
sql = f"""
SELECT
    stkcd,
    accper,
    b002000000 / a003000000 AS roe,
    b002000000 / b001100000 AS profitmargin,
    b001100000 / a001000000 AS turnover,
    a001000000 / a003000000 AS leverage
FROM csmar.wrds_csmar_financial_master
WHERE stkcd IN {stock_list}
  AND typrep = 'A'
"""

# Execute query and load data
data = db.raw_sql(sql, date_cols=["accper"])

# Data cleaning 
df = data[data["accper"].dt.month == 12].copy()
df["year"] = df["accper"].dt.year
df = df[df["year"] >= START_YEAR].copy()
df = df.drop(columns=["accper"])
df["Company"] = df["stkcd"].map(COMPANIES)

# ==============================================
# Step 3: Calculate Pivot Tables for Financial Ratios
# ==============================================

# ROE pivot table 
roe_pivot = (
    df.pivot(index="Company", columns="year", values="roe")
    .mul(100).round(1).astype(str) + "%"
).rename_axis(columns=None).rename_axis("ROE (%)", axis="index")

# Net Profit Margin pivot table 
pm_pivot = (
    df.pivot(index="Company", columns="year", values="profitmargin")
    .mul(100).round(1).astype(str) + "%"
).rename_axis(columns=None).rename_axis("Profit Margin (%)", axis="index")

# Asset Turnover pivot table 
to_pivot = (
    df.pivot(index="Company", columns="year", values="turnover")
    .mul(100).round(1).astype(str) + "%"
).rename_axis(columns=None).rename_axis("Asset Turnover (%)", axis="index")

# Financial Leverage pivot table
lev_pivot = (
    df.pivot(index="Company", columns="year", values="leverage")
    .round(2)
).rename_axis(columns=None).rename_axis("Leverage", axis="index")

# Print results 
print("=== Final Financial Ratio Analysis Results ===")
print("\n--- ROE (Return on Equity) ---")
print(roe_pivot.to_string())
print("\n--- Net Profit Margin ---")
print(pm_pivot.to_string())
print("\n--- Asset Turnover ---")
print(to_pivot.to_string())
print("\n--- Financial Leverage ---")
print(lev_pivot.to_string())

# ==============================================
# Step 4: Descriptive Analysis of Financial Ratios
# ==============================================

# Calculate 5-year average for each ratio (core descriptive statistics)
desc_stats = df.groupby('Company').agg({
    'roe': ['mean', 'min', 'max'],
    'profitmargin': ['mean', 'min', 'max'],
    'turnover': ['mean', 'min', 'max'],
    'leverage': ['mean', 'min', 'max']
}).round(4) * 100  

# Rename columns for readability
desc_stats.columns = ['ROE_Mean(%)', 'ROE_Min(%)', 'ROE_Max(%)',
                      'ProfitMargin_Mean(%)', 'ProfitMargin_Min(%)', 'ProfitMargin_Max(%)',
                      'Turnover_Mean(%)', 'Turnover_Min(%)', 'Turnover_Max(%)',
                      'Leverage_Mean', 'Leverage_Min', 'Leverage_Max']

print("=== Descriptive Analysis Summary (2020-2024) ===")
print("\n5-Year Key Statistics by Company:")
print(desc_stats.to_string())

# ==============================================
# Step 5: Export Final Results to CSV & Excel
# ==============================================
df.to_csv("financial_ratios.csv", index=False, encoding="utf-8-sig")
df.to_excel("financial_ratios.xlsx", index=False, engine="openpyxl")
print("\n✅ CSV and Excel files exported successfully!")

# ==============================================
# Step 6: close
# ==============================================
db.close()

Loading library list...
Done
=== Final Financial Ratio Analysis Results ===

--- ROE (Return on Equity) ---
            2020   2021   2022   2023   2024
ROE (%)                                     
BYD         9.3%   3.8%  14.6%  20.8%  20.9%
Changan     6.1%   6.5%  12.3%  12.7%   7.7%
Great Wall  9.4%  10.8%  12.7%  10.3%  16.1%

--- Net Profit Margin ---
                   2020  2021  2022  2023  2024
Profit Margin (%)                              
BYD                3.8%  1.8%  4.2%  5.2%  5.4%
Changan            3.9%  3.4%  6.4%  6.3%  3.8%
Great Wall         5.2%  4.9%  6.0%  4.1%  6.3%

--- Asset Turnover ---
                     2020   2021   2022   2023   2024
Asset Turnover (%)                                   
BYD                 77.9%  73.1%  85.9%  88.6%  99.2%
Changan             69.9%  77.7%  83.0%  79.6%  76.7%
Great Wall          67.1%  77.8%  74.1%  86.1%  93.1%

--- Financial Leverage ---
            2020  2021  2022  2023  2024
Leverage                             

### Descriptive Analysis Insights
#### 1. Profitability Performance
- **BYD**: Achieved the highest 5-year average ROE (13.88%), with a maximum of 20.9% in 2024, showing the strongest long-term profitability growth driven by scale expansion.
- **Great Wall**: Maintained a stable average ROE of 11.86%, with consistent net profit margin (average 5.22%), reflecting reliable operational profitability.
- **Changan**: Had the lowest average ROE (9.26%) and the largest fluctuation, indicating higher operational instability compared to peers.

#### 2. Operational Efficiency
- **BYD**: Led the industry with an average asset turnover of 87.74%, reaching 99.2% in 2024, demonstrating exceptional asset utilization efficiency.
- **Changan**: Showed significant improvement, with average turnover rising from 69.9% to 93.1%, reflecting enhanced operational management.
- **Great Wall**: Maintained a stable average turnover of 79.06%, with consistent operational performance.

#### 3. Financial Risk Profile
- **BYD**: Adopted a moderate leverage strategy (average 3.68), balancing growth expansion and financial risk.
- **Changan & Great Wall**: Used conservative leverage (averages 2.43 and 2.83 respectively), prioritizing risk control over aggressive growth.

This descriptive analysis transforms raw financial data into actionable insights, laying a solid foundation for follow-up visualization and conclusion derivation.